python -m ipykernel install --user --name=job_scraper_env --display-name="Job Scraper venv"

jupyter kernelspec list


## DB에서 데이터 가져오기 

In [139]:
import pandas as pd 
from job_scraper.database.connection import db_manager, init_database
from job_scraper.database.models import JobPosting

In [140]:
init_database()

with db_manager.get_session() as session:
    postings = session.query(JobPosting).all()
    data = []
    for job in postings:
        data.append({
            'experience': job.experience,
            'position': job.position
        })
    
    df = pd.DataFrame(data)

12:43:35 - [INFO] - job_scraper.database.connection - 🔌 Initializing database connection...
12:43:35 - [INFO] - job_scraper.database.connection - ✅ Database connection initialized: localhost:5432/job_scraper_db
12:43:35 - [INFO] - job_scraper.database.connection - 🏗️ Creating database tables...
12:43:35 - [INFO] - job_scraper.database.connection - ✅ Database tables created successfully


In [141]:
len(df['position'].unique())

571

[Engineering] 
- AI 엔지니어 or AI Engineer  | ML 엔지니어 | Machine Learning Engineer 
- LLM 엔지니어 
- AI agent 엔지니어 
- MLOps Engineer 
- AI 개발자 

[Data]
- Data Engineer or 데이터 엔지니어 
- Data Scientist or 데이터 사이언티스트
- Data Analytics Engineer or DA or 데이터 분석가 

[R&D]
- AI Researcher or AI 연구원 

In [142]:
position = [k for k, v in df['position'].value_counts().items()]

In [143]:
position[11]

'데이터 엔지니어 (3년 이상)'

In [144]:
position

['데이터 엔지니어',
 'AI 엔지니어',
 'Data Engineer',
 'AI Engineer',
 'Data Scientist',
 'DBA',
 'AI 개발자',
 '백엔드 개발자',
 'Senior AI Research Engineer',
 'ML Engineer',
 'Machine Learning Engineer',
 '데이터 엔지니어 (3년 이상)',
 'Senior ML Engineer',
 '데이터 엔지니어 (Data Engineer)',
 'AI Researcher',
 'AI Software Engineer',
 'AI Agent Engineer',
 'LLM Research Engineer',
 'MLOps Engineer',
 '머신러닝 백엔드 개발자',
 'LLM Engineer',
 'Data Analytics Engineer',
 'AI 연구원',
 'Data Analyst (시니어)',
 '금융/NL2SQL 데이터 사이언티스트',
 '[쿠팡] Staff Data Scientist (Biz Analytics)',
 'Data Engineer (PM)',
 'Data Scientist (팀원)',
 '데이터 사이언티스트',
 'Data Analyst (5년 이상)',
 'Data Analyst',
 '딥러닝 엔지니어 (1 ~ 5년)',
 '[DATA&AI] 데이터 사이언티스트',
 '콘텐츠 프로듀서 [AI 엔지니어링 교육]',
 '[바이오던스] AI 트랜스포메이션 매니저',
 '임상 통계 분석 담당자',
 '[공통] Data Analytics Engineer',
 'Data Scientist(3년이상)',
 '[공통] Data Scientist',
 'DataOps Manager (Data Analyst)',
 'SCM 컨설팅(Python 기반)',
 'Data Architect & Healthcare Data Engineer',
 '[Tech] Data Architect - 모델 진단 및 거버넌스 구축',
 '데이터 사이언티스

In [145]:
import re 

In [146]:
def clean_bracket(text): 
    all_bracket_pattern = r"[(\[]([^)\]]*)[)\]]"
    result = re.sub(all_bracket_pattern, '', text)
    
    return result

def remove_dash_descriptions(text):
    """하이픈 뒤 설명 제거 (신중하게)"""
    # 안전한 제거 패턴들
    safe_remove_patterns = [
        r'\s*-\s*전문연구요원.*',
        r'\s*-\s*\d+년.*경력.*',
        r'\s*-\s*신입.*',
        r'\s*-\s*경력.*',
        r'\s*-\s*근무지.*',
        r'\s*-\s*모델\s+진단.*',  # "모델 진단 및 거버넌스 구축" 같은 부가설명
    ]
    
    for pattern in safe_remove_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    return text

def normalize_slash_terms(text):
    """슬래시 기반 용어 정규화"""
    # 중복/유사 기술 용어 정리
    redundant_patterns = {
        r'생성형\s*AI/LLM': 'LLM',
        r'AI/LLM': 'LLM', 
        r'ML/머신러닝': 'ML',
        r'머신러닝/ML': 'ML',
        r'백엔드/ML': 'Backend ML',
        r'프론트엔드·백엔드': 'Fullstack',
        r'Frontend/Backend': 'Fullstack',
    }
    
    for pattern, replacement in redundant_patterns.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    
    # 유의미한 슬래시는 보존 (기술 스택)
    # 예: "웹 크롤링/스크래핑", "NLP/LLM" 등
    
    return text

def remove_experience_terms(text):
    """경력/레벨 표현 제거"""
    experience_patterns = [
        r'\d+~\d+년차',
        r'\d+-\d+년차', 
        r'신입~초급',
        r'주니어|Junior|Jr\.',
        r'시니어|Senior|Sr\.',
        r'리더|Leader',
        r'리드|Lead',
    ]
    
    for pattern in experience_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    return text


def clean_job_title(text):
    """단계별 직무명 정리"""
    result = text.strip()
    
    # 1. 소괄호/대괄호 포함 문자 제거 
    result = clean_bracket(result)
    
    # 2. 하이픈/대시 뒤 부가설명 제거 (조건부)
    result = remove_dash_descriptions(result)
    
    # 3. 슬래시 기반 문자열 정리
    result = normalize_slash_terms(result)
    
    # 4. 경력/레벨 표현 제거
    result = remove_experience_terms(result)
   
    # 5. 미처리된 괄호/대괄호 잔여물 제거
    result = re.sub(r'[\(\)\[\]]+', '', result)
    
    # 6. 최종 공백 정리
    result = re.sub(r'\s{2,}', ' ', result).strip()
    
    return result

In [147]:
for name in position: 
    print("원본: " + name, " | ", "수정본: " + clean_job_title(name))

원본: 데이터 엔지니어  |  수정본: 데이터 엔지니어
원본: AI 엔지니어  |  수정본: AI 엔지니어
원본: Data Engineer  |  수정본: Data Engineer
원본: AI Engineer  |  수정본: AI Engineer
원본: Data Scientist  |  수정본: Data Scientist
원본: DBA  |  수정본: DBA
원본: AI 개발자  |  수정본: AI 개발자
원본: 백엔드 개발자  |  수정본: 백엔드 개발자
원본: Senior AI Research Engineer  |  수정본: AI Research Engineer
원본: ML Engineer  |  수정본: ML Engineer
원본: Machine Learning Engineer  |  수정본: Machine Learning Engineer
원본: 데이터 엔지니어 (3년 이상)  |  수정본: 데이터 엔지니어
원본: Senior ML Engineer  |  수정본: ML Engineer
원본: 데이터 엔지니어 (Data Engineer)  |  수정본: 데이터 엔지니어
원본: AI Researcher  |  수정본: AI Researcher
원본: AI Software Engineer  |  수정본: AI Software Engineer
원본: AI Agent Engineer  |  수정본: AI Agent Engineer
원본: LLM Research Engineer  |  수정본: LLM Research Engineer
원본: MLOps Engineer  |  수정본: MLOps Engineer
원본: 머신러닝 백엔드 개발자  |  수정본: 머신러닝 백엔드 개발자
원본: LLM Engineer  |  수정본: LLM Engineer
원본: Data Analytics Engineer  |  수정본: Data Analytics Engineer
원본: AI 연구원  |  수정본: AI 연구원
원본: Data Analyst (시니어)  |  수정본: Data 

In [148]:
len(set(clean_job_title(name) for name in position))

408

In [149]:
# JOB_NORMALIZATION_RULES = {
#     '데이터 엔지니어': [
#         'Data Engineer', 'Data 엔지니어', '데이터 Engineer',
#         'DE', 'Data Infrastructure Engineer'
#     ],
    
#     'AI 엔지니어': [
#         'AI Engineer', 'AI Developer', 'AI 개발자',
#         'ML Engineer', 'Machine Learning Engineer', 'ML 엔지니어',
#         '머신러닝 Engineer', 'Machine Learning Developer',
#         'Deep Learning Engineer', 'DL Engineer', '머신러닝 엔지니어'
#     ],
    
#     '데이터 사이언티스트': [
#         'Data Scientist', 'Data 사이언티스트', '데이터 Scientist',
#         'DS', 'Data Science', 'Applied Scientist'
#     ],
    
#     '데이터 분석가': [
#         'Data Analyst', 'Data 분석가', '데이터 Analyst',
#         'Business Analyst', 'DA', 'Data Analytics'
#     ],
    
#     'AI 연구원': [
#         'AI Researcher', 'ML Researcher',
#         'Machine Learning Researcher', 'Research Scientist',
#         'AI Research Engineer'
#     ],
    
#     '벡엔드 개발자': [
#         'Backend Developer', 'Backend Engineer', '백엔드 Developer',
#         'Server Developer', 'API Developer'
#     ],
    
#     '프론트엔드 개발자': [
#         'Frontend Developer', 'Frontend Engineer', '프론트엔드 Developer',
#         'Web Frontend Developer', 'UI Developer'
#     ],
    
#     '풀스택 개발자': [
#         'Fullstack Developer', 'Full Stack Developer', 'Full-Stack Developer',
#         'Fullstack Engineer', '풀스택 Developer'
#     ]
# }

JOB_NORMALIZATION_RULES = {
    '데이터 엔지니어': [
        'data engineer', 'data 엔지니어', '데이터 engineer',
        'de', 'data infrastructure engineer'
    ],
    
    'AI 엔지니어': [
        'ai engineer', 'ai developer', 'ai 개발자',
        'ml engineer', 'machine learning engineer', 'ml 엔지니어',
        '머신러닝 engineer', 'machine learning developer',
        'deep learning engineer', 'dl engineer', '머신러닝 엔지니어'
    ],
    
    '데이터 사이언티스트': [
        'data scientist', 'data 사이언티스트', '데이터 scientist',
        'ds', 'data science', 'applied scientist'
    ],
    
    '데이터 분석가': [
        'data analyst', 'data 분석가', '데이터 analyst',
        'business analyst', 'da', 'ba', 'data analytics'
    ],
    
    'AI 연구원': [
        'ai researcher', 'ml researcher',
        'machine learning researcher', 'research scientist',
        'ai research engineer'
    ],
    
    '벡엔드 개발자': [
        'backend developer', 'backend engineer', '백엔드 developer',
        'server developer', 'api developer'
    ],
    
    '프론트엔드 개발자': [
        'frontend developer', 'frontend engineer', '프론트엔드 developer',
        'web frontend developer', 'ui developer'
    ],
    
    '풀스택 개발자': [
        'fullstack developer', 'full stack developer', 'full-stack developer',
        'fullstack engineer', '풀스택 developer'
    ]
}

In [150]:
normed_position = [clean_job_title(name) for name in position]

In [151]:
test = 'Data Eng'
test.lower()

'data eng'

In [152]:
def normalize_by_rules(text): 
    lower_text = text.lower()
    
    for standard_name, name_list in JOB_NORMALIZATION_RULES.items(): 
        if lower_text in name_list: 
            return standard_name 
        
    return text       

In [153]:
len([normalize_by_rules(name) for name in normed_position])

571

In [154]:
len(set([normalize_by_rules(name) for name in normed_position]))

387

In [137]:
for name in normed_position: 
    print(name, ' || ', normalize_by_rules(name))

데이터 엔지니어  ||  데이터 엔지니어
AI 엔지니어  ||  AI 엔지니어
Data Engineer  ||  데이터 엔지니어
AI Engineer  ||  AI 엔지니어
Data Scientist  ||  데이터 사이언티스트
DBA  ||  DBA
AI 개발자  ||  AI 엔지니어
백엔드 개발자  ||  백엔드 개발자
AI Research Engineer  ||  AI 연구원
ML Engineer  ||  AI 엔지니어
Machine Learning Engineer  ||  AI 엔지니어
데이터 엔지니어  ||  데이터 엔지니어
ML Engineer  ||  AI 엔지니어
데이터 엔지니어  ||  데이터 엔지니어
AI Researcher  ||  AI 연구원
AI Software Engineer  ||  AI Software Engineer
AI Agent Engineer  ||  AI Agent Engineer
LLM Research Engineer  ||  LLM Research Engineer
MLOps Engineer  ||  MLOps Engineer
머신러닝 백엔드 개발자  ||  머신러닝 백엔드 개발자
LLM Engineer  ||  LLM Engineer
Data Analytics Engineer  ||  Data Analytics Engineer
AI 연구원  ||  AI 연구원
Data Analyst  ||  데이터 분석가
금융/NL2SQL 데이터 사이언티스트  ||  금융/NL2SQL 데이터 사이언티스트
Staff Data Scientist  ||  Staff Data Scientist
Data Engineer  ||  데이터 엔지니어
Data Scientist  ||  데이터 사이언티스트
데이터 사이언티스트  ||  데이터 사이언티스트
Data Analyst  ||  데이터 분석가
Data Analyst  ||  데이터 분석가
딥러닝 엔지니어  ||  딥러닝 엔지니어
데이터 사이언티스트  ||  데이터 사이언티스트
콘텐츠 프로듀서  

In [124]:
[normalize_by_rules(name) for name in normed_position]

['데이터 엔지니어',
 'ai 엔지니어',
 '데이터 엔지니어',
 'ai engineer',
 'data scientist',
 'dba',
 'ai 개발자',
 '백엔드 개발자',
 'ai research engineer',
 'ml engineer',
 'machine learning engineer',
 '데이터 엔지니어',
 'ml engineer',
 '데이터 엔지니어',
 'ai researcher',
 'ai software engineer',
 'ai agent engineer',
 'llm research engineer',
 'mlops engineer',
 '머신러닝 백엔드 개발자',
 'llm engineer',
 'data analytics engineer',
 'ai 연구원',
 'data analyst',
 '금융/nl2sql 데이터 사이언티스트',
 'staff data scientist',
 '데이터 엔지니어',
 'data scientist',
 '데이터 사이언티스트',
 'data analyst',
 'data analyst',
 '딥러닝 엔지니어',
 '데이터 사이언티스트',
 '콘텐츠 프로듀서',
 'ai 트랜스포메이션 매니저',
 '임상 통계 분석 담당자',
 'data analytics engineer',
 'data scientist',
 'data scientist',
 'dataops manager',
 'scm 컨설팅',
 'data architect & healthcare data engineer',
 'data architect',
 '데이터 사이언티스트',
 'operation / digital consultant',
 'prompt engineer',
 'ml backend engineer',
 '검색 엔지니어',
 '맵 데이터 분석 담당자',
 'data analyst',
 'data scientist',
 '데이터 사이언티스트',
 'ds',
 '광학 엔지니어',
 '데이터 기획 및 분석',
 'd

In [100]:
# raw_title	원본 직무명
# standardized_title	알고리즘이 판단한 표준 직무
# confidence	변환에 대한 신뢰 점수 (예: 0.75 이상이면 자동 수락)
# matching_method	rule_based, pattern, similarity 등 구분값
# manual_review	사람이 수동 확인했는지 여부 (True/False)
# correction	수동으로 수정했을 경우 정답 필드
# review_note	사람이 판단 시 코멘트 저장 가능하도록
# created_at / reviewed_at	추후 타임라인 추적용 필드

In [ ]:
from dataclasses import dataclass

@dataclass
class  

class JobGrouper: 
    def __init__(self): 
        self.rule_bases = None 
        self.pattern = None 
        self.similarity = None 

    def standard_job_titles(self, job_names): 
        if self.rule_bases: 
            grouped_jobs = self.rule_bases(job_names)
            
        return grouped_jobs
    
    def _preprocess_job_name(self): 
        pass 